In [ ]:
!pip install wandb thop albumentations==1.3.1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import numpy as np
import wandb
import albumentations as A
from albumentations.pytorch import ToTensorV2
from thop import profile
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
config = {
    "project_name": "cifar-10",
    "epochs": 30,
    "batch_size": 512,        # Larger batch size for faster GPU throughput
    "max_lr": 0.01,           # Peak LR for OneCycle
    "weight_decay": 1e-4,
    "grad_clip": 0.1,         # Gradient clipping
    "architecture": "ResNet9-Custom",
}

# --- 1. PRO DATASET & DATALOADER (Albumentations) ---
class ProCIFAR10(Dataset):
    def __init__(self, root, train=True, download=True):
        self.dataset = torchvision.datasets.CIFAR10(root=root, train=train, download=download)
        self.data = self.dataset.data
        self.targets = self.dataset.targets

        # Industry standard augmentations (Cutout, Shifts, Flips)
        if train:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
                A.CoarseDropout(max_holes=1, max_height=16, max_width=16,
                                min_holes=1, min_height=16, min_width=16,
                                fill_value=[0.4914*255, 0.4822*255, 0.4465*255], p=0.5),
                A.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)),
                ToTensorV2(),
            ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]
        label = self.targets[idx]
        # Albumentations requires numpy images
        augmented = self.transform(image=image)["image"]
        return augmented, label

# --- 2. MODEL: CUSTOM RESNET-9 (High Perf/Low Params) ---
def conv_block(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if pool: layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

class ResNet9(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        # Initial Prep
        self.conv1 = conv_block(in_channels, 64)
        self.conv2 = conv_block(64, 128, pool=True)

        # ResBlock 1
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128))

        # Mid Layers
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True)

        # ResBlock 2
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512))

        # Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveMaxPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.res1(out) + out  # Residual Connection
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out  # Residual Connection
        out = self.classifier(out)
        return out

# --- 3. ANALYSIS UTILS (Hooks & FLOPs) ---
grad_stats = {}

def save_grad_stats(name):
    """Hook to capture gradient norms during backprop"""
    def hook(grad):
        # Calculate L2 norm of the gradient for this layer
        grad_norm = grad.norm().item()
        if name not in grad_stats:
            grad_stats[name] = []
        grad_stats[name].append(grad_norm)
    return hook

def register_hooks(model):
    """Registers hooks to all Conv layers"""
    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            # Register backward hook on the tensor output of the layer usually,
            # but simpler to register on parameters for weight updates.
            # Here we register on parameters to get exact weight gradient.
            pass

# --- 4. MAIN EXECUTION ---
def train_and_analyze():
    wandb.init(project=config['project_name'], config=config)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Init Data
    train_ds = ProCIFAR10('./data', train=True)
    test_ds = ProCIFAR10('./data', train=False)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=config['batch_size']*2, shuffle=False, num_workers=2)

    # Init Model
    model = ResNet9().to(device)

    # FLOP Counting
    inp = torch.randn(1, 3, 32, 32).to(device)
    macs, params = profile(model, inputs=(inp,), verbose=False)
    print(f"Stats: {params/1e6:.2f}M Params | {macs/1e6:.2f}M FLOPs")
    wandb.log({"Params (M)": params/1e6, "FLOPs (M)": macs/1e6})

    # Optimizer & Scheduler (OneCycle - The "Pro" Choice)
    optimizer = optim.AdamW(model.parameters(), lr=config['max_lr'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=config['max_lr'],
                                              epochs=config['epochs'],
                                              steps_per_epoch=len(train_loader))

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # Label smoothing for regularization

    # Training Loop
    wandb.watch(model, log="all", log_freq=50) # Auto-logs gradients/parameters histograms

    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0

        # Snapshot weights for update flow
        old_weights = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()

            # Gradient Clipping
            nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            _, pred = output.max(1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

        # --- MANUAL VISUALIZATION LOGIC ---
        # 1. Gradient Flow (Average gradient per layer for this epoch)
        # We manually inspect .grad attributes after the last batch
        layers = []
        grads = []
        for n, p in model.named_parameters():
            if (p.requires_grad) and ("bias" not in n) and (p.grad is not None):
                layers.append(n.replace(".weight", ""))
                grads.append(p.grad.abs().mean().item())

        grad_data = [[x, y] for (x, y) in zip(layers, grads)]
        table_grad = wandb.Table(data=grad_data, columns=["Layer", "Gradient Norm"])

        # 2. Weight Update Flow (How much did weights move?)
        new_weights = {n: p.detach() for n, p in model.named_parameters() if p.requires_grad}
        update_layers = []
        update_mags = []
        for n, p_new in new_weights.items():
            if n in old_weights and "bias" not in n:
                # Calculate Frobenius norm of difference / Frobenius norm of original (Relative change)
                diff = (p_new - old_weights[n]).norm().item()
                orig = old_weights[n].norm().item()
                if orig > 0:
                    update_layers.append(n.replace(".weight", ""))
                    update_mags.append(diff / orig) # Relative update

        update_data = [[x, y] for (x, y) in zip(update_layers, update_mags)]
        table_update = wandb.Table(data=update_data, columns=["Layer", "Relative Update"])

        # Validation
        model.eval()
        val_acc = 0
        with torch.no_grad():
            val_correct = 0
            val_total = 0
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                out = model(data)
                _, pred = out.max(1)
                val_correct += pred.eq(target).sum().item()
                val_total += target.size(0)
            val_acc = 100. * val_correct / val_total

        # Log Everything
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": epoch_loss / len(train_loader),
            "train_acc": 100. * correct / total,
            "val_acc": val_acc,
            "lr": scheduler.get_last_lr()[0],
            "gradient_flow": wandb.plot.bar(table_grad, "Layer", "Gradient Norm", title="Gradient Flow (End of Epoch)"),
            "weight_updates": wandb.plot.line(table_update, "Layer", "Relative Update", title="Relative Weight Updates")
        })

        print(f"Epoch {epoch+1} | Train Acc: {100.*correct/total:.2f}% | Val Acc: {val_acc:.2f}%")

    wandb.finish()

# Run it
train_and_analyze()

Stats: 6.57M Params | 380.77M FLOPs
Epoch 1 | Train Acc: 41.64% | Val Acc: 58.75%
Epoch 2 | Train Acc: 57.81% | Val Acc: 63.23%
Epoch 3 | Train Acc: 63.68% | Val Acc: 71.30%
Epoch 4 | Train Acc: 67.57% | Val Acc: 69.53%
Epoch 5 | Train Acc: 69.02% | Val Acc: 72.42%
Epoch 6 | Train Acc: 70.21% | Val Acc: 66.31%
Epoch 7 | Train Acc: 71.23% | Val Acc: 64.47%
Epoch 8 | Train Acc: 73.52% | Val Acc: 62.42%
Epoch 9 | Train Acc: 76.44% | Val Acc: 78.51%
Epoch 10 | Train Acc: 78.92% | Val Acc: 82.05%
Epoch 11 | Train Acc: 80.81% | Val Acc: 84.13%
Epoch 12 | Train Acc: 82.20% | Val Acc: 84.50%
Epoch 13 | Train Acc: 83.90% | Val Acc: 84.23%
Epoch 14 | Train Acc: 85.02% | Val Acc: 86.12%
Epoch 15 | Train Acc: 86.10% | Val Acc: 87.95%
Epoch 16 | Train Acc: 87.40% | Val Acc: 87.06%
Epoch 17 | Train Acc: 88.35% | Val Acc: 89.06%
Epoch 18 | Train Acc: 89.23% | Val Acc: 90.25%
Epoch 19 | Train Acc: 89.96% | Val Acc: 88.43%
Epoch 20 | Train Acc: 90.94% | Val Acc: 91.44%
Epoch 21 | Train Acc: 91.66% | Va

FLOPs (M),▁
Params (M),▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,▁▂▃▄▅▆▇█████▇▇▇▆▆▅▅▄▄▃▃▂▂▂▁▁▁▁
train_acc,▁▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██████████
train_loss,█▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▄▃▄▃▂▂▅▆▆▆▆▇▇▇▇▇▇███████████
FLOPs (M),380.76928
Params (M),6.57313
epoch,30
lr,0.0
